In [16]:
from nltk.corpus import wordnet as wn

In [17]:
import json
from datetime import datetime
import numpy as np
def jsonPretify(object_list):
    # Custom serializer to convert datetime and numpy objects
    def datetime_handler(obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, np.ndarray):
            # Print a clean metadata summary instead of raw binary elements
            return f"<ndarray: shape={obj.shape}, dtype={obj.dtype}>"
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")
        
    # Serialize using the custom handler
    pretty_json = json.dumps(object_list, indent=4, sort_keys=True, default=datetime_handler)
    return pretty_json

In [18]:


# import os
# from datasets import load_dataset

# # 1. Path to your local cache folder containing '1aurent___ade20_k'
# local_cache_dir = os.path.abspath("../datasets/ade20k")

# 2. Load the dataset (it will read from your local .arrow files without downloading again)
# dataset = load_dataset("1aurent/ADE20K", cache_dir=local_cache_dir)
# dataset = load_dataset("uva-cv-lab/ADE20k-150", cache_dir=local_cache_dir)
import os
import sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
import expanded_benchmark_helpers as bm_hp
COCO_ANN  = os.path.join(REPO_ROOT, "datasets", "coco", "instances_val2017.json")
ADE20K_PATH = os.path.join(REPO_ROOT, "datasets", "ade20k")
ade_dataset = bm_hp.load_ade20k(ADE20K_PATH)
coco_dataset = bm_hp.load_coco(COCO_ANN)




Loaded ADE20K validation dataset from local cache.
loading annotations into memory...
Done (t=0.21s)
creating index...
index created!


In [19]:
print(ade_dataset[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x39C224070>, 'segmentations': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=256x256 at 0x39C224EE0>], 'instances': [<PIL.PngImagePlugin.PngImageFile image mode=L size=256x256 at 0x39C224970>, <PIL.PngImagePlugin.PngImageFile image mode=L size=256x256 at 0x39C224AF0>, <PIL.PngImagePlugin.PngImageFile image mode=L size=256x256 at 0x39C224BE0>, <PIL.PngImagePlugin.PngImageFile image mode=L size=256x256 at 0x39C224CA0>, <PIL.PngImagePlugin.PngImageFile image mode=L size=256x256 at 0x39C224130>, <PIL.PngImagePlugin.PngImageFile image mode=L size=256x256 at 0x39C224FD0>, <PIL.PngImagePlugin.PngImageFile image mode=L size=256x256 at 0x39C224460>], 'filename': 'ADE_val_00000025.jpg', 'folder': 'ADE20K_2021_17_01/images/ADE/validation/cultural/apse__indoor', 'source': {'folder': 'static_sun_database/a/apse/indoor', 'filename': 'sun_ajoswmtbceygukcq.jpg', 'origin': 'Downloaded from Google search. Image might be 

### BenchMark

In [8]:
from sentence_transformers import SentenceTransformer, util
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device='mps') # Use 'cpu' or 'cuda' as needed

Coco

In [6]:
best_synset_id, data, definition = bm_hp.get_best_synset_id_frm_bn('person', 'person')
print(best_synset_id)
# data = bm_hp._babelnet_get_synset_data(best_synset_id)
print(data)
print(definition)

bn:16076822n
{'senses': [{'type': 'BabelSense', 'properties': {'fullLemma': 'Person_(surname)', 'simpleLemma': 'person', 'lemma': {'lemma': 'Person', 'type': 'HIGH_QUALITY'}, 'source': 'WIKI', 'senseKey': '39933035', 'frequency': 0, 'language': 'EN', 'pos': 'NOUN', 'synsetID': {'id': 'bn:16076822n', 'pos': 'NOUN', 'source': 'BABELNET'}, 'translationInfo': '', 'pronunciations': {'audios': [], 'transcriptions': []}, 'bKeySense': False, 'idSense': 101167643, 'tags': {'class it.uniroma1.lcl.babelnet.data.LabelTag': [{'language': 'EN', 'label': 'surname'}]}}}, {'type': 'BabelSense', 'properties': {'fullLemma': 'Person', 'simpleLemma': 'person', 'lemma': {'lemma': 'Person', 'type': 'HIGH_QUALITY'}, 'source': 'WIKIDATA', 'senseKey': 'Q16881132', 'frequency': 0, 'language': 'EN', 'pos': 'NOUN', 'synsetID': {'id': 'bn:16076822n', 'pos': 'NOUN', 'source': 'BABELNET'}, 'translationInfo': '', 'pronunciations': {'audios': [], 'transcriptions': []}, 'bKeySense': False, 'idSense': 496579724, 'tags': 

In [21]:
synset_ids = bm_hp._babelnet_get_synset_ids('bowl')
context_emb = embedding_model.encode('kitchen', convert_to_tensor=True)
context_txt = 'kitchen'
for sid in synset_ids[:5]:
    print(sid)
    data = bm_hp._babelnet_get_synset_data(sid)
    definition = ""
    for gloss in data.get("glosses", []):
        if gloss.get("language") == "EN":
            definition = gloss.get("gloss", "")
            break
    # candidate_text = f"{definition} related to {context_txt}"
    candidate_emb = embedding_model.encode(definition, convert_to_tensor=True)
    score = util.cos_sim(context_emb, candidate_emb).item()
    print(score)
    print(definition)
    w_s = bm_hp._babelnet_extract_english_lemmas(data)
    w_s_synset = bm_hp._babelnet_extract_english_lemmas(bm_hp._babelnet_get_synset_data(sid))
    print(w_s)
    print(w_s_synset)
    print()


bn:14620277n
0.18378393352031708
Kashkul also referred to as the beggar's bowl is a container carried by wandering Dervishes and used to collect money and other goods
['kashkul', 'shell_bowl', 'bowl', 'shell']
['kashkul', 'shell_bowl', 'bowl', 'shell']

bn:08145359n
0.2663971185684204

['bowl']
['bowl']

bn:00012500n
0.1893303096294403
The act of rolling something (as the ball in bowling)
['roll', 'bowl']
['roll', 'bowl']

bn:00012499n
0.22099699079990387
A small round container that is open at the top for holding tobacco
['bowl', 'pipe_bowl', 'marijuana_pipe']
['bowl', 'pipe_bowl', 'marijuana_pipe']

bn:00012498n
0.08300337195396423
A wooden ball (with flattened sides so that it rolls on a curved course) used in the game of lawn bowling
['bowl']
['bowl']



In [35]:
from nltk.corpus import wordnet as wn
def clean_taxonomy_list(raw_list):
    """
    Cleans a list of taxnomy words by:
      1. Splitting comma-separated strings into individual elements.
      2. Normalizing spaces to underscores and converting to lowercase.
      3. Filtering out misspelled words/typos using WordNet.
      4. Deduplicating while preserving order.
    """
    cleaned = []
    for item in raw_list:
        # Split by comma (handles "altar, communion table, Lord's table" -> ['altar', 'communion table', "Lord's table"])
        parts = [p.strip().lower() for p in item.split(",")]
        
        for part in parts:
            # Replace spaces with underscores
            part_normalized = part.replace(" ", "_")
            if not part_normalized:
                continue
                
            # Check if it's already in the cleaned list
            if part_normalized not in cleaned:
                # Use WordNet to filter out typos/misspellings
                # (e.g. "fa\u00e7ade" or "botle" will return [] and be skipped)
                if wn.synsets(part_normalized) or wn.synsets(part.replace("_", " ")):
                    cleaned.append(part_normalized)
                    
    return cleaned

In [36]:
from pycocotools import mask as maskUtils
from time import time

#command to download caoc dataset
# mkdir -p datasets/coco
# curl -L -o datasets/coco/instances_val2017.json http://images.cocodataset.org/annotations/annotations_trainval2017.zip
# unzip datasets/coco/instances_val2017.json -d datasets/coco/

def create_benchmark_from_coco(coco_dataset, limit=5000, img_bnch=[], cat_bnch={}):
    img_ids = coco_dataset.getImgIds()
    print(len(img_ids))
    for img_id in img_ids[:limit]:
        img_meta = coco_dataset.loadImgs(img_id)[0]
        h, w = img_meta["height"], img_meta["width"]

        gt_objects = []
        gt_bin_masks = []
        anns = bm_hp.get_anns_for_img_id(img_id, coco_dataset)
        
        for ann in anns:
            start_time = time()
            category = coco_dataset.loadCats(ann["category_id"])[0]
            # print(category)
            cat_name = category["name"]
            if cat_name not in gt_objects:
                gt_objects.append(cat_name)
                gt_bin_masks.append(bm_hp.get_gt_mask(coco_dataset, img_id, category['id']))
                # gt_bin_masks.append(get_bin_masks(ann["segmentation"], h, w))
            if cat_name not in cat_bnch:
                # 1. Query local WordNet first (extremely fast)
                definition_wn, w_s_wn, w_s_hp_wn, w_s_he_wn = bm_hp.build_word_sets_from_synset_v2(
                    word=cat_name, supporting_words=category["supercategory"]
                )
                definition = definition_wn
                synonyms = w_s_wn
                hyponyms = w_s_hp_wn
                hypernyms = w_s_he_wn
                
                # 2. Fallback to BabelNet ONLY if WordNet fails or has fewer than 2 synonyms
                if not w_s_wn or len(w_s_wn) < 2:
                    print(f"WordNet failed/insufficient for '{cat_name}'. Querying BabelNet API...")
                    definition_bn, w_s_bn, w_s_hp_bn, w_s_he_bn = bm_hp.supplement_word_sets_with_babelnet_v2(
                        word=cat_name, supporting_words=category["supercategory"]
                    )
                
                definition = definition_bn if definition_wn == "" else definition_wn
                synonyms.extend(w_s_bn)
                hyponyms.extend(w_s_hp_bn)
                hypernyms.extend(w_s_he_wn)
                
                
                # 3. Save to cat_bnch, limiting lists to at most 5 unique elements
                cat_bnch[cat_name].append({
                    "cat_src_id" : category["id"],
                    "cat_src" : "coco",
                    "cat_id" : f"coco_{category['id']}",
                    "definition" : definition,
                    "synonyms": clean_taxonomy_list(synonyms[:5]),
                    "hyponyms": clean_taxonomy_list(hyponyms[:5]),
                    "hypernyms": clean_taxonomy_list(hypernyms[:5])
                })
            print(f"time taken: {(time()-start_time)*1000:.1f} ms")
        
        img_bnch.append({
            "img_src" : "coco",
            "img_src_id": img_id,
            "img_id" : f"coco_{img_id}",
            "width": w,
            "height": h,
            "filename": img_meta["file_name"],
            "img_url": img_meta.get("coco_url", ""),
            "gt_objects": gt_objects,
            "gt_bin_masks": gt_bin_masks
        })

    return img_bnch, cat_bnch
    
    


In [37]:
import numpy as np
from time import time
from PIL import Image

def create_benchmark_from_ade20K(ade_dataset, limit=2000, img_bnch=[], cat_bnch={}):
    for idx in range(limit):
        print(f"Processing dataset[{idx+1}]...")
        data = ade_dataset[idx]
        img_pil = data["image"]
        img_id = int(os.path.splitext(data['filename'])[0].split('_')[-1])
        w, h = img_pil.size

        gt_objects = []
        gt_bin_masks = []
        
        for obj_id,obj in enumerate(data["objects"]):
            start_time = time()
            # print(category)
            cat_name = obj["raw_name"]
            if cat_name not in gt_objects:
                gt_objects.append(cat_name)
                gt_bin_masks.append(bm_hp.get_gt_mask_for_ade(data, cat_name))
            if cat_name not in cat_bnch:
                # 1. Query local WordNet first (extremely fast)
                hypernyms = obj["hypernym"] if len(obj["hypernym"]) > 0 else []
                definition_wn, w_s_wn, w_s_hp_wn, w_s_he_wn = bm_hp.build_word_sets_from_synset_v2(
                    word=cat_name, supporting_words=obj["hypernym"][:2]
                )
                definition = definition_wn
                synonyms = w_s_wn
                hyponyms = w_s_hp_wn
                hypernyms.extend(w_s_he_wn)
                
                # 2. Fallback to BabelNet ONLY if WordNet fails or has fewer than 2 synonyms
                if not w_s_wn:
                    print(f"WordNet failed/insufficient for '{cat_name}'. Querying BabelNet API...")
                    definition_bn, w_s_bn, w_s_hp_bn, w_s_he_bn = bm_hp.supplement_word_sets_with_babelnet_v2(
                        word=cat_name, supporting_words=obj["hypernym"][:2]
                    )
                
                    definition = definition_bn if definition_wn == "" else definition_wn
                    synonyms.extend(w_s_bn)
                    hyponyms.extend(w_s_hp_bn)
                    hypernyms.extend(w_s_he_wn)
                
                
                # 3. Save to cat_bnch, limiting lists to at most 5 unique elements
                cat_bnch[cat_name].append({
                    "cat_src_id" : obj['name_ndx'],
                    "cat_src" : "ade20k",
                    "cat_id" : f"ade20k_{obj['name_ndx']}",
                    "definition" : definition,
                    "synonyms": clean_taxonomy_list(synonyms)[:5],
                    "hyponyms": clean_taxonomy_list(hyponyms)[:5],
                    "hypernyms": clean_taxonomy_list(hypernyms)[:5]
                })
            print(f"time taken: {(time()-start_time)*1000:.1f} ms")
        
        img_bnch.append({
            "img_src" : "ade20k",
            "img_src_id": img_id,
            "img_id" : f"ade20k_{img_id}",
            "width": w,
            "height": h,
            "filename": data["filename"],
            "img_url": "",
            "gt_objects": gt_objects,
            "gt_bin_masks": gt_bin_masks
        })
        print("Processing completed!!")

    return img_bnch, cat_bnch

In [39]:
from collections import defaultdict
img_bnch = []
cat_bnch = defaultdict(list)
img_c, cat_c = create_benchmark_from_coco(coco_dataset, 5, img_bnch, cat_bnch)
img, cat = create_benchmark_from_ade20K(ade_dataset, 5, img_c, cat_c)

5000
WordNet failed/insufficient for 'bottle'. Querying BabelNet API...
time taken: 544.6 ms
time taken: 1.3 ms
WordNet failed/insufficient for 'person'. Querying BabelNet API...
time taken: 235.2 ms
time taken: 89.4 ms
WordNet failed/insufficient for 'bowl'. Querying BabelNet API...
time taken: 291.5 ms
time taken: 0.0 ms
WordNet failed/insufficient for 'oven'. Querying BabelNet API...
time taken: 65.6 ms
time taken: 0.0 ms
WordNet failed/insufficient for 'cup'. Querying BabelNet API...
time taken: 306.2 ms
time taken: 0.0 ms
time taken: 0.0 ms
time taken: 0.0 ms
WordNet failed/insufficient for 'broccoli'. Querying BabelNet API...
time taken: 138.8 ms
WordNet failed/insufficient for 'spoon'. Querying BabelNet API...
time taken: 205.3 ms
time taken: 0.0 ms
time taken: 0.0 ms
time taken: 0.0 ms
WordNet failed/insufficient for 'carrot'. Querying BabelNet API...
time taken: 215.0 ms
WordNet failed/insufficient for 'sink'. Querying BabelNet API...
time taken: 207.6 ms
WordNet failed/insuff

In [ ]:
import numpy as np
from PIL import Image
from transformers.image_utils import load_image
from pycocotools.mask import frPyObjects, decode


def load_image_lazy(img_info, dataset= None):
    """
    Loads PIL Image on-the-fly without saving local image files.
    """
    if img_info["img_src"] == "coco":
        # Load directly from public COCO URL into RAM
        return load_image(img_info["img_url"])
    elif img_info["img_src"] == "ade20k":
        # Pull from the memory-mapped HF dataset
        if dataset is None:
            raise ValueError("ade_dataset must be provided to resolve ADE20K images")
        return dataset[img_info["img_src_id"]]["image"]
    raise ValueError("Unknown image source")

img = load_image_lazy(img_bnch[0])

In [16]:
img.show()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [40]:
print(jsonPretify(img))

[
    {
        "filename": "000000397133.jpg",
        "gt_bin_masks": [
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>",
            "<ndarray: shape=(427, 640), dtype=uint8>"
        ],
        "gt_objects": [
            "bottle",
            "dining table",
            "person",
            "knife",
            "bowl",
            "oven",
            "cup",
            "broccoli",
            "spoon",
            "carrot",
            "sink"
        ],
        "height": 427,
    

In [41]:
print(jsonPretify(cat))

{
    "air conditioning": [
        {
            "cat_id": "ade20k_10",
            "cat_src": "ade20k",
            "cat_src_id": 10,
            "definition": "a system that keeps air cool and dry",
            "hypernyms": [
                "air_conditioner",
                "air_conditioning",
                "cooling_system",
                "cooling",
                "mechanism"
            ],
            "hyponyms": [],
            "synonyms": [
                "air_conditioner",
                "air_conditioning"
            ]
        }
    ],
    "altar": [
        {
            "cat_id": "ade20k_3108",
            "cat_src": "ade20k",
            "cat_src_id": 3108,
            "definition": "the table in Christian churches where communion is given",
            "hypernyms": [
                "altar",
                "communion_table",
                "lord's_table",
                "table",
                "furniture"
            ],
            "hyponyms": [],
            "

In [30]:
from collections import defaultdict
def build_pos_neg_sets(img_bnch):
    """
    Builds positive and negative image sets for categories in the benchmark.
    
    Returns:
      positive_set: {category: [img_ids where category exists]}
      negative_set: {category: [img_ids where category does NOT exist]}
    """
    # 1. Collect all unique image IDs in this benchmark
    all_img_ids = [img["img_id"] for img in img_bnch]
    all_img_ids_set = set(all_img_ids)
    
    # 2. Build the positive set
    positive_set = defaultdict(list)
    for img in img_bnch:
        img_id = img["img_id"]
        for cat in img["gt_objects"]:
            if img_id not in positive_set[cat]:
                positive_set[cat].append(img_id)
                
    # Sort positive lists for consistency
    positive_set = {cat: sorted(ids) for cat, ids in positive_set.items()}
    
    # 3. Build the negative set (All IDs minus positive IDs)
    negative_set = {}
    for cat, pos_ids in positive_set.items():
        pos_set = set(pos_ids)
        negative_set[cat] = sorted(list(all_img_ids_set - pos_set))
        
    return positive_set, negative_set

In [42]:
# Combine both benchmarks into a single list

pos_combined, neg_combined = build_pos_neg_sets(img)

print(jsonPretify(pos_combined))


{
    "air conditioning": [
        "ade20k_1029"
    ],
    "altar": [
        "ade20k_25"
    ],
    "aquarium": [
        "ade20k_26"
    ],
    "balustrade": [
        "ade20k_25"
    ],
    "banana": [
        "coco_37777"
    ],
    "bicycle": [
        "coco_174482",
        "coco_87038"
    ],
    "book": [
        "ade20k_1029"
    ],
    "bookcase": [
        "ade20k_29"
    ],
    "books": [
        "ade20k_29"
    ],
    "bottle": [
        "coco_397133"
    ],
    "bowl": [
        "coco_397133"
    ],
    "box": [
        "ade20k_1029"
    ],
    "boxes": [
        "ade20k_29"
    ],
    "broccoli": [
        "coco_397133"
    ],
    "button": [
        "ade20k_1028"
    ],
    "button panel": [
        "ade20k_1028"
    ],
    "buttons": [
        "ade20k_1028"
    ],
    "cabinet": [
        "ade20k_1028",
        "ade20k_1029"
    ],
    "car": [
        "coco_174482"
    ],
    "carrot": [
        "coco_397133"
    ],
    "ceiling": [
        "ade20k_1028",
        "a

In [43]:
print(jsonPretify(neg_combined))


{
    "air conditioning": [
        "ade20k_1028",
        "ade20k_25",
        "ade20k_26",
        "ade20k_29",
        "coco_174482",
        "coco_252219",
        "coco_37777",
        "coco_397133",
        "coco_87038"
    ],
    "altar": [
        "ade20k_1028",
        "ade20k_1029",
        "ade20k_26",
        "ade20k_29",
        "coco_174482",
        "coco_252219",
        "coco_37777",
        "coco_397133",
        "coco_87038"
    ],
    "aquarium": [
        "ade20k_1028",
        "ade20k_1029",
        "ade20k_25",
        "ade20k_29",
        "coco_174482",
        "coco_252219",
        "coco_37777",
        "coco_397133",
        "coco_87038"
    ],
    "balustrade": [
        "ade20k_1028",
        "ade20k_1029",
        "ade20k_26",
        "ade20k_29",
        "coco_174482",
        "coco_252219",
        "coco_37777",
        "coco_397133",
        "coco_87038"
    ],
    "banana": [
        "ade20k_1028",
        "ade20k_1029",
        "ade20k_25",
        "a